# Redes Recurrentes y Transformers — Ejemplos Prácticos
**Curso:** Machine Learning — Universidad Distrital FJC  
**Profesor:** Ing. Carlos Andrés Sierra, M.Sc.

Este notebook complementa el módulo de Deep Learning (secuencias). Contiene cuatro secciones prácticas:
1. **LSTM para series de tiempo** — pronóstico de temperatura mínima diaria (dataset descargado con `curl`)
2. **RNN, GRU y LSTM** — comparación sobre la misma serie + un modelo de lenguaje a nivel de caracteres
3. **Seq2Seq con GRU + atención de Bahdanau** — traducción español → inglés con fragmentos de *Cien años de soledad*
4. **Transformer desde cero** — la misma traducción con atención multi-cabeza (*scaled dot-product attention*)

> **Requisitos:** `torch numpy pandas matplotlib` y `curl` en el sistema  
> Instala con: `pip install torch numpy pandas matplotlib`

In [1]:
# Imports globales y configuración
import os, math, random, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

ModuleNotFoundError: No module named 'torch'

---
## Sección 1 — LSTM para Series de Tiempo: Temperatura Mínima Diaria
*(Redes recurrentes, memoria de largo plazo)*

**Problema:** dada la temperatura mínima de los últimos `W` días en Melbourne (Australia, 1981–1990), predecir la del día siguiente.  
**Dataset:** *Daily Minimum Temperatures in Melbourne* (3 650 observaciones, Australian Bureau of Meteorology), disponible públicamente en el repositorio de Jason Brownlee.

Descargamos el archivo con **curl** directamente desde el notebook.

In [ ]:
os.makedirs('./data', exist_ok=True)
URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'

# -sS: silencioso pero muestra errores | -L: sigue redirecciones | -o: archivo de salida
if not os.path.exists('./data/daily-min-temperatures.csv'):
    subprocess.run(['curl', '-sSL', '-o', './data/daily-min-temperatures.csv', URL], check=True)

# Equivalente en una celda de Jupyter:  !curl -sSL -o ./data/daily-min-temperatures.csv $URL
df = pd.read_csv('./data/daily-min-temperatures.csv', parse_dates=['Date'], index_col='Date')
df['Temp'] = pd.to_numeric(df['Temp'], errors='coerce').interpolate()   # el CSV original trae algún valor sucio
print(df.shape); print(df.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['Temp'].plot(ax=axes[0], lw=0.6)
axes[0].set_title('Temperatura mínima diaria — Melbourne'); axes[0].set_ylabel('°C'); axes[0].grid(True, alpha=0.3)
df['Temp']['1981-01':'1981-12'].plot(ax=axes[1])
axes[1].set_title('Zoom: 1981 (estacionalidad anual)'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Preparación de datos: ventanas deslizantes

- **Split cronológico** 80 % / 20 % (nunca se mezcla el futuro con el pasado).
- **Normalización** con media y desviación del conjunto de entrenamiento únicamente (evita *data leakage*).
- Cada muestra: ventana de `W = 30` días → la salida es el valor del día 31.

In [ ]:
W = 30
values = df['Temp'].values.astype(np.float32)
n_train = int(len(values) * 0.8)
mu, sigma = values[:n_train].mean(), values[:n_train].std()
scaled = (values - mu) / sigma

def make_windows(series, w):
    X = np.stack([series[i:i + w] for i in range(len(series) - w)])
    y = series[w:]
    return X[..., None], y            # X: (N, w, 1)  y: (N,)

X_all, y_all = make_windows(scaled, W)
split = n_train - W                    # el índice de la muestra i corresponde al objetivo i+W
Xtr, ytr = map(torch.tensor, (X_all[:split], y_all[:split]))
Xte, yte = map(torch.tensor, (X_all[split:], y_all[split:]))
print(f'Train: {tuple(Xtr.shape)} | Test: {tuple(Xte.shape)}')

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

### El modelo: LSTM → capa lineal

Usamos el **último estado oculto** de la secuencia para predecir el siguiente valor.  
Definimos una clase genérica `RNNForecaster` donde `kind` puede ser `'RNN'`, `'GRU'` o `'LSTM'` (la reutilizaremos en la Sección 2).

In [ ]:
class RNNForecaster(nn.Module):
    def __init__(self, kind='LSTM', hidden=64, n_layers=2, dropout=0.1):
        super().__init__()
        rnn_cls = {'RNN': nn.RNN, 'GRU': nn.GRU, 'LSTM': nn.LSTM}[kind]
        self.rnn = rnn_cls(1, hidden, n_layers, batch_first=True, dropout=dropout)
        self.fc  = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)              # out: (B, W, hidden)
        return self.fc(out[:, -1]).squeeze(-1)

def fit_forecaster(model, epochs=15, lr=1e-3, verbose=True):
    model = model.to(DEVICE)
    opt, crit = optim.Adam(model.parameters(), lr=lr), nn.MSELoss()
    hist = {'train': [], 'test': []}
    for ep in range(epochs):
        model.train(); tl = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # evita gradientes explosivos
            opt.step(); tl += loss.item()
        model.eval()
        with torch.no_grad():
            te = crit(model(Xte.to(DEVICE)), yte.to(DEVICE)).item()
        hist['train'].append(tl / len(train_loader)); hist['test'].append(te)
        if verbose: print(f'  Época {ep+1:2d}/{epochs} — train MSE: {hist["train"][-1]:.4f} | test MSE: {te:.4f}')
    return model, hist

lstm_model, lstm_hist = fit_forecaster(RNNForecaster('LSTM'))

In [ ]:
def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X.to(DEVICE)).cpu().numpy() * sigma + mu      # de vuelta a °C

y_true = yte.numpy() * sigma + mu
y_lstm = predict(lstm_model, Xte)
y_naive = Xte[:, -1, 0].numpy() * sigma + mu                       # línea base: "mañana = hoy"

def metrics(y, yhat): return np.mean(np.abs(y - yhat)), np.sqrt(np.mean((y - yhat) ** 2))
for name, yh in [('Persistencia (ayer)', y_naive), ('LSTM', y_lstm)]:
    mae, rmse = metrics(y_true, yh)
    print(f'{name:22s} MAE: {mae:.3f} °C | RMSE: {rmse:.3f} °C')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(lstm_hist['train'], 'b-o', label='Train'); axes[0].plot(lstm_hist['test'], 'r-s', label='Test')
axes[0].set_title('Curva de aprendizaje (MSE normalizado)'); axes[0].set_xlabel('Época')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

t = df.index[n_train:]
k = 150
axes[1].plot(t[:k], y_true[:k], 'k-', label='Real')
axes[1].plot(t[:k], y_lstm[:k], 'r--', label='LSTM (1 día adelante)')
axes[1].set_title('Pronóstico a un día — primeros 150 días de test'); axes[1].set_ylabel('°C')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Pronóstico recursivo (multi-paso)

Para predecir **30 días hacia adelante** realimentamos las predicciones del modelo como entrada.  
Observa cómo el error se **acumula** y la predicción tiende a suavizarse hacia la media estacional.

In [ ]:
H = 30
start = 100                                       # ventana de test desde la cual pronosticamos
window = Xte[start].clone().numpy()               # (W, 1)
preds = []
lstm_model.eval()
with torch.no_grad():
    for _ in range(H):
        p = lstm_model(torch.tensor(window[None]).to(DEVICE)).item()
        preds.append(p)
        window = np.vstack([window[1:], np.array([[p]], dtype=np.float32)])   # desplaza la ventana e incluye la predicción

preds = np.array(preds) * sigma + mu
real  = y_true[start:start + H]
plt.figure(figsize=(9, 4))
plt.plot(range(1, H + 1), real,  'k-o', ms=3, label='Real')
plt.plot(range(1, H + 1), preds, 'r--s', ms=3, label='LSTM recursivo')
plt.xlabel('Días hacia adelante'); plt.ylabel('°C'); plt.title(f'Pronóstico a {H} días')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'MAE a {H} días: {np.mean(np.abs(real - preds)):.2f} °C')

---
## Sección 2 — RNN vs GRU vs LSTM (y un modelo de lenguaje a nivel de carácter)

### 2a. Comparación de celdas recurrentes
Entrenamos las tres variantes con la **misma arquitectura y datos**, pero con ventanas largas (`W = 90`)  
para que la memoria de largo plazo importe. Recuerda: la RNN simple sufre *vanishing gradients*; GRU y LSTM usan compuertas.

In [ ]:
W2 = 90
X2, y2 = make_windows(scaled, W2)
sp2 = n_train - W2
Xtr, ytr = map(torch.tensor, (X2[:sp2], y2[:sp2]))      # reutiliza las variables globales que usa fit_forecaster
Xte, yte = map(torch.tensor, (X2[sp2:], y2[sp2:]))
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
y_true2 = yte.numpy() * sigma + mu

results = {}
for kind in ['RNN', 'GRU', 'LSTM']:
    torch.manual_seed(SEED)
    print(f'\n== {kind} ==')
    m, h = fit_forecaster(RNNForecaster(kind), epochs=12, verbose=False)
    mae, rmse = metrics(y_true2, predict(m, Xte))
    n_par = sum(p.numel() for p in m.parameters())
    results[kind] = dict(hist=h, mae=mae, rmse=rmse, params=n_par)
    print(f'  Parámetros: {n_par:,} | MAE: {mae:.3f} °C | RMSE: {rmse:.3f} °C')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for kind, c in zip(results, ['tab:orange', 'tab:green', 'tab:blue']):
    axes[0].plot(results[kind]['hist']['test'], '-o', color=c, label=kind)
axes[0].set_title('MSE en test por época (W = 90)'); axes[0].set_xlabel('Época')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].bar(results.keys(), [r['mae'] for r in results.values()], color=['tab:orange', 'tab:green', 'tab:blue'])
axes[1].set_title('MAE final en test (°C)'); axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

> **Discusión:** en esta serie el patrón relevante está sobre todo en los últimos días, así que las diferencias entre celdas
> son pequeñas y pueden cambiar con la semilla. Las compuertas se vuelven decisivas cuando la dependencia es **larga**
> (texto, audio, señales con eventos distantes). Se sugiere subir `W2` a 200–365 y compara.

### 2b. Modelo de lenguaje a nivel de carácter con GRU
Una RNN también puede **generar** secuencias: predice el siguiente carácter dado el contexto.  
Entrenamos con el comienzo de *Cien años de soledad* (corpus diminuto → el modelo memoriza; el objetivo es ver el mecanismo).

In [ ]:
corpus = (
    "Muchos años después, frente al pelotón de fusilamiento, el coronel Aureliano Buendía había de recordar "
    "aquella tarde remota en que su padre lo llevó a conocer el hielo. Macondo era entonces una aldea de veinte "
    "casas de barro y cañabrava construidas a la orilla de un río de aguas diáfanas que se precipitaban por un "
    "lecho de piedras pulidas, blancas y enormes como huevos prehistóricos. El mundo era tan reciente, que muchas "
    "cosas carecían de nombre, y para mencionarlas había que señalarlas con el dedo. "
)
chars = sorted(set(corpus)); c2i = {c: i for i, c in enumerate(chars)}; i2c = {i: c for c, i in c2i.items()}
data = torch.tensor([c2i[c] for c in corpus])
print(f'Caracteres: {len(corpus)} | Vocabulario: {len(chars)}')

class CharGRU(nn.Module):
    def __init__(self, V, E=32, H=128):
        super().__init__()
        self.emb = nn.Embedding(V, E); self.gru = nn.GRU(E, H, 2, batch_first=True); self.fc = nn.Linear(H, V)
    def forward(self, x, h=None):
        out, h = self.gru(self.emb(x), h)
        return self.fc(out), h

SEQ = 40
starts = torch.arange(0, len(data) - SEQ - 1)
Xc = torch.stack([data[s:s + SEQ] for s in starts]); Yc = torch.stack([data[s + 1:s + SEQ + 1] for s in starts])
char_loader = DataLoader(TensorDataset(Xc, Yc), batch_size=64, shuffle=True)

torch.manual_seed(SEED)
cm = CharGRU(len(chars)).to(DEVICE)
opt = optim.Adam(cm.parameters(), lr=3e-3)
losses = []
for ep in range(60):
    tl = 0
    for xb, yb in char_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits, _ = cm(xb)
        loss = F.cross_entropy(logits.reshape(-1, len(chars)), yb.reshape(-1))
        loss.backward(); nn.utils.clip_grad_norm_(cm.parameters(), 1.0); opt.step(); tl += loss.item()
    losses.append(tl / len(char_loader))
    if (ep + 1) % 10 == 0: print(f'  Época {ep+1}/60 — Loss: {losses[-1]:.4f}')

In [ ]:
@torch.no_grad()
def generate(model, seed, n=150, temperature=0.8):
    model.eval()
    x = torch.tensor([[c2i[c] for c in seed]]).to(DEVICE)
    logits, h = model(x)
    out = list(seed)
    for _ in range(n):
        probs = F.softmax(logits[0, -1] / temperature, dim=-1)
        idx = torch.multinomial(probs, 1).item()
        out.append(i2c[idx])
        logits, h = model(torch.tensor([[idx]]).to(DEVICE), h)
    return ''.join(out)

# EXPERIMENTO: temperature baja → texto conservador/repetitivo; alta → más "creativo" (y más errores)
for T in [0.3, 0.8, 1.5]:
    print(f'--- temperature = {T} ---')
    print(generate(cm, 'Macondo era', temperature=T), '\n')

---
## Sección 3 — Seq2Seq con GRU + Atención de Bahdanau: Español → Inglés
*(Fragmentos de **Cien años de soledad**, Gabriel García Márquez)*

**Arquitectura:**
- **Encoder:** GRU bidireccional → produce un vector por palabra de origen (*annotations*).
- **Atención aditiva (Bahdanau):** en cada paso del decoder calcula un **vector de atención** $\alpha_t$ sobre las palabras de origen y un *vector de contexto* $c_t=\sum_i \alpha_{t,i}h_i$.
- **Decoder:** GRU que recibe la palabra previa y $c_t$ para producir la siguiente palabra.

$$e_{t,i}=v^\top\tanh(W s_{t-1}+U h_i),\qquad \alpha_{t,i}=\mathrm{softmax}_i(e_{t,i})$$

> **Nota sobre los datos:** el corpus tiene ~40 oraciones cortas, tomadas o adaptadas de fragmentos del libro, con traducciones al inglés escritas
> para este ejercicio (no son la traducción publicada de Gregory Rabassa). Es un corpus *reducido*: sirve para entender el mecanismo y
> ver las matrices de atención, **no** para obtener un traductor real. Reservamos 4 oraciones para mostrar qué pasa con texto no visto.

In [ ]:
pairs = [
 ("Muchos años después, frente al pelotón de fusilamiento, el coronel Aureliano Buendía había de recordar aquella tarde remota en que su padre lo llevó a conocer el hielo.",
  "Many years later, facing the firing squad, Colonel Aureliano Buendia was to remember that distant afternoon when his father took him to discover ice."),
 ("Macondo era entonces una aldea de veinte casas de barro y cañabrava.",
  "Macondo was then a village of twenty houses made of mud and reeds."),
 ("El mundo era tan reciente que muchas cosas carecían de nombre.",
  "The world was so recent that many things lacked a name."),
 ("Para mencionarlas había que señalarlas con el dedo.",
  "To mention them one had to point at them with a finger."),
 ("Todos los años una familia de gitanos plantaba su carpa cerca de la aldea.",
  "Every year a family of gypsies set up their tent near the village."),
 ("Los gitanos llevaban inventos nuevos a Macondo.",
  "The gypsies brought new inventions to Macondo."),
 ("Melquíades era un gitano corpulento con barba de gorrión.",
  "Melquiades was a stout gypsy with a sparrow's beard."),
 ("José Arcadio Buendía era un hombre emprendedor.",
  "Jose Arcadio Buendia was an enterprising man."),
 ("José Arcadio Buendía nunca había visto el hielo.",
  "Jose Arcadio Buendia had never seen ice."),
 ("El hielo era el invento más grande de nuestro tiempo.",
  "Ice was the greatest invention of our time."),
 ("Úrsula era una mujer de carácter fuerte.",
  "Ursula was a woman of strong character."),
 ("Úrsula cuidaba la casa con mucha paciencia.",
  "Ursula took care of the house with great patience."),
 ("La casa era grande y tenía muchas habitaciones.",
  "The house was big and had many rooms."),
 ("Aureliano pasaba el día en el taller de platería.",
  "Aureliano spent the day in the silversmith workshop."),
 ("El coronel hizo treinta y dos guerras y las perdió todas.",
  "The colonel fought thirty-two wars and lost them all."),
 ("El coronel esperaba una carta que nunca llegaba.",
  "The colonel was waiting for a letter that never arrived."),
 ("Remedios la bella subió al cielo en cuerpo y alma.",
  "Remedios the beauty rose to heaven in body and soul."),
 ("Remedios la bella no conocía la malicia.",
  "Remedios the beauty did not know malice."),
 ("Llovió durante cuatro años, once meses y dos días.",
  "It rained for four years, eleven months and two days."),
 ("Las flores amarillas caían del cielo como una lluvia silenciosa.",
  "Yellow flowers fell from the sky like a silent rain."),
 ("La peste del insomnio llegó al pueblo.",
  "The plague of insomnia arrived in the town."),
 ("Los habitantes olvidaban el nombre de las cosas.",
  "The inhabitants forgot the names of things."),
 ("Pilar Ternera leía el futuro en las cartas.",
  "Pilar Ternera read the future in the cards."),
 ("Amaranta tejía su mortaja con paciencia.",
  "Amaranta wove her shroud with patience."),
 ("Arcadio fue fusilado al amanecer.",
  "Arcadio was shot at dawn."),
 ("El pueblo estaba rodeado de montañas y de agua.",
  "The town was surrounded by mountains and water."),
 ("El tren llegó a Macondo con un silbido largo.",
  "The train arrived in Macondo with a long whistle."),
 ("La compañía bananera cambió la vida del pueblo.",
  "The banana company changed the life of the town."),
 ("Los soldados dispararon contra la multitud en la estación.",
  "The soldiers fired at the crowd in the station."),
 ("Nadie recordaba la matanza al día siguiente.",
  "Nobody remembered the massacre the next day."),
 ("Fernanda llegó de una ciudad lejana y fría.",
  "Fernanda came from a distant and cold city."),
 ("La soledad seguía a la familia durante cien años.",
  "Solitude followed the family for a hundred years."),
 ("Aureliano descifró los pergaminos de Melquíades.",
  "Aureliano deciphered the parchments of Melquiades."),
 ("Los pergaminos contaban la historia de la familia.",
  "The parchments told the story of the family."),
 ("Macondo desapareció con el viento.",
  "Macondo disappeared with the wind."),
 ("Las estirpes condenadas a cien años de soledad no tenían una segunda oportunidad sobre la tierra.",
  "Lines condemned to one hundred years of solitude did not have a second opportunity on earth."),
 # --- reservadas para test (no se ven en entrenamiento) ---
 ("Úrsula cuidaba la casa durante cien años.", "Ursula took care of the house for a hundred years."),
 ("El coronel esperaba el hielo.", "The colonel was waiting for the ice."),
 ("Los gitanos llegaron al pueblo con un tren.", "The gypsies arrived in the town with a train."),
 ("Macondo era una aldea de montañas y de agua.", "Macondo was a village of mountains and water."),
]
train_pairs, test_pairs = pairs[:-4], pairs[-4:]
print(f'Pares de entrenamiento: {len(train_pairs)} | Pares de prueba (no vistos): {len(test_pairs)}')

In [ ]:
import re
PAD, SOS, EOS, UNK = 0, 1, 2, 3

def tokenize(s):
    # minúsculas + separa signos de puntuación como tokens
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", s.lower())

class Vocab:
    def __init__(self, sentences):
        self.itos = ['<pad>', '<sos>', '<eos>', '<unk>'] + sorted({w for s in sentences for w in tokenize(s)})
        self.stoi = {w: i for i, w in enumerate(self.itos)}
    def encode(self, s, bos_eos=False):
        ids = [self.stoi.get(w, UNK) for w in tokenize(s)]
        return [SOS] + ids + [EOS] if bos_eos else ids + [EOS]
    def decode(self, ids):
        return ' '.join(self.itos[i] for i in ids if i not in (PAD, SOS, EOS))
    def __len__(self): return len(self.itos)

src_vocab = Vocab([p[0] for p in train_pairs])
tgt_vocab = Vocab([p[1] for p in train_pairs])
print(f'Vocabulario ES: {len(src_vocab)} | EN: {len(tgt_vocab)}')

def pad(seqs):
    L = max(map(len, seqs))
    return torch.tensor([s + [PAD] * (L - len(s)) for s in seqs])

SRC = pad([src_vocab.encode(p[0]) for p in train_pairs])                    # (N, S)   termina en <eos>
TGT = pad([tgt_vocab.encode(p[1], bos_eos=True) for p in train_pairs])      # (N, T)   <sos> ... <eos>
print('SRC:', tuple(SRC.shape), '| TGT:', tuple(TGT.shape))

In [ ]:
class Encoder(nn.Module):
    def __init__(self, V, E=64, H=128):
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=PAD)
        self.rnn = nn.GRU(E, H, batch_first=True, bidirectional=True)
        self.fc  = nn.Linear(2 * H, H)
    def forward(self, src):
        out, h = self.rnn(self.emb(src))                              # out: (B, S, 2H)
        h0 = torch.tanh(self.fc(torch.cat([h[0], h[1]], dim=1)))      # estado inicial del decoder
        return out, h0

class BahdanauAttention(nn.Module):
    def __init__(self, H):
        super().__init__()
        self.W = nn.Linear(H, H, bias=False)       # proyecta el estado del decoder
        self.U = nn.Linear(2 * H, H, bias=False)   # proyecta las annotations del encoder
        self.v = nn.Linear(H, 1, bias=False)
    def forward(self, s, enc_out, mask):
        e = self.v(torch.tanh(self.W(s).unsqueeze(1) + self.U(enc_out))).squeeze(-1)   # (B, S)
        alpha = F.softmax(e.masked_fill(~mask, -1e9), dim=-1)                          # vector de atención
        ctx = torch.bmm(alpha.unsqueeze(1), enc_out).squeeze(1)                        # (B, 2H)
        return ctx, alpha

class AttnDecoder(nn.Module):
    def __init__(self, V, E=64, H=128):
        super().__init__()
        self.emb  = nn.Embedding(V, E, padding_idx=PAD)
        self.attn = BahdanauAttention(H)
        self.cell = nn.GRUCell(E + 2 * H, H)
        self.out  = nn.Linear(H + 2 * H + E, V)
        self.drop = nn.Dropout(0.1)
    def step(self, y, s, enc_out, mask):
        e = self.drop(self.emb(y))
        ctx, alpha = self.attn(s, enc_out, mask)
        s = self.cell(torch.cat([e, ctx], dim=1), s)
        return self.out(torch.cat([s, ctx, e], dim=1)), s, alpha

class Seq2SeqAttn(nn.Module):
    def __init__(self, Vs, Vt, E=64, H=128):
        super().__init__()
        self.enc, self.dec = Encoder(Vs, E, H), AttnDecoder(Vt, E, H)
    def forward(self, src, tgt):
        mask = src != PAD
        enc_out, s = self.enc(src)
        logits = []
        for t in range(tgt.size(1) - 1):                    # teacher forcing
            lg, s, _ = self.dec.step(tgt[:, t], s, enc_out, mask)
            logits.append(lg)
        return torch.stack(logits, dim=1)                   # (B, T-1, V)

    @torch.no_grad()
    def translate(self, sentence, vocab_s, vocab_t, max_len=40):
        self.eval()
        src = torch.tensor([vocab_s.encode(sentence)]).to(DEVICE)
        mask = src != PAD
        enc_out, s = self.enc(src)
        y = torch.tensor([SOS]).to(DEVICE); out, attns = [], []
        for _ in range(max_len):
            lg, s, a = self.dec.step(y, s, enc_out, mask)
            y = lg.argmax(-1)
            if y.item() == EOS: break
            out.append(y.item()); attns.append(a[0].cpu().numpy())
        return vocab_t.decode(out), np.array(attns)

torch.manual_seed(SEED)
s2s = Seq2SeqAttn(len(src_vocab), len(tgt_vocab)).to(DEVICE)
print(f'Parámetros: {sum(p.numel() for p in s2s.parameters()):,}')

In [ ]:
def train_translator(model, epochs, lr, bs=16, forward_fn=None):
    opt = optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(SRC, TGT), batch_size=bs, shuffle=True)
    losses = []
    for ep in range(epochs):
        model.train(); tl = 0
        for sb, tb in loader:
            sb, tb = sb.to(DEVICE), tb.to(DEVICE)
            opt.zero_grad()
            logits = model(sb, tb)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tb[:, 1:].reshape(-1), ignore_index=PAD)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tl += loss.item()
        losses.append(tl / len(loader))
        if (ep + 1) % max(1, epochs // 6) == 0: print(f'  Época {ep+1}/{epochs} — Loss: {losses[-1]:.4f}')
    return losses

s2s_losses = train_translator(s2s, epochs=120, lr=2e-3)

In [ ]:
def show_translations(model, pairs_, title):
    print(f'=== {title} ===')
    for es, en in pairs_:
        pred, _ = model.translate(es, src_vocab, tgt_vocab)
        print(f'ES : {es}\nREF: {en.lower()}\nPRED: {pred}\n')

show_translations(s2s, train_pairs[:4], 'Oraciones de ENTRENAMIENTO')
show_translations(s2s, test_pairs, 'Oraciones NO VISTAS')

In [ ]:
def plot_attention(attn, src_tokens, tgt_tokens, title, ax=None):
    ax = ax or plt.gca()
    im = ax.imshow(attn, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens, rotation=60, ha='right')
    ax.set_yticks(range(len(tgt_tokens))); ax.set_yticklabels(tgt_tokens)
    ax.set_xlabel('Español (origen)'); ax.set_ylabel('Inglés (generado)'); ax.set_title(title)
    return im

sent = "Remedios la bella subió al cielo en cuerpo y alma."
pred, A = s2s.translate(sent, src_vocab, tgt_vocab)
plt.figure(figsize=(8, 5))
im = plot_attention(A, tokenize(sent) + ['<eos>'], pred.split(), 'Atención de Bahdanau (GRU)')
plt.colorbar(im); plt.tight_layout(); plt.show()
print('Traducción:', pred)

> **Lectura del mapa:** cada fila es una palabra generada en inglés y cada columna una palabra en español.
> Un color intenso indica que el decoder consultó esa palabra de origen al generarla.
>
> **Observación:** con tan pocas oraciones el modelo memoriza la traducción completa y la atención suele concentrarse
> en una o dos columnas (con frecuencia las últimas, cuyas annotations ya resumen la oración entera), en lugar de formar
> una alineación palabra a palabra. Los mapas interpretables aparecen con corpus grandes; compárese con las cabezas de
> atención cruzada del Transformer en la Sección 4, donde sí se observan correspondencias léxicas (por ejemplo,
> *body / and / soul* con *cuerpo / y / alma*).

---
## Sección 4 — Transformer desde Cero: Español → Inglés
*(Attention Is All You Need, Vaswani et al., 2017)*

Eliminamos la recurrencia: toda la secuencia se procesa **en paralelo** con **auto-atención**.  
Implementamos manualmente la atención escalada multi-cabeza:

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Componentes: embeddings + codificación posicional sinusoidal → *N* capas de encoder (auto-atención + FFN) →
*N* capas de decoder (auto-atención **enmascarada** + atención cruzada + FFN).  
Reutilizamos los mismos datos y vocabularios de la Sección 3 para poder comparar.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(d_model, d_model) for _ in range(4))
        self.attn = None                                   # guardamos los pesos para visualizarlos

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        split = lambda x, lin: lin(x).view(B, -1, self.h, self.dk).transpose(1, 2)    # (B, h, T, dk)
        Q, K, V = split(q, self.wq), split(k, self.wk), split(v, self.wv)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)                          # (B, h, Tq, Tk)
        if mask is not None: scores = scores.masked_fill(~mask, -1e9)
        self.attn = F.softmax(scores, dim=-1)
        out = (self.attn @ V).transpose(1, 2).reshape(B, -1, self.h * self.dk)
        return self.wo(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class FFN(nn.Module):
    def __init__(self, d, dff, p):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d, dff), nn.ReLU(), nn.Dropout(p), nn.Linear(dff, d))
    def forward(self, x): return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d, h, dff, p):
        super().__init__()
        self.sa, self.ff = MultiHeadAttention(d, h), FFN(d, dff, p)
        self.n1, self.n2, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(p)
    def forward(self, x, mask):
        x = self.n1(x + self.drop(self.sa(x, x, x, mask)))
        return self.n2(x + self.drop(self.ff(x)))

class DecoderLayer(nn.Module):
    def __init__(self, d, h, dff, p):
        super().__init__()
        self.sa, self.ca, self.ff = MultiHeadAttention(d, h), MultiHeadAttention(d, h), FFN(d, dff, p)
        self.n1, self.n2, self.n3, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(p)
    def forward(self, y, enc, self_mask, cross_mask):
        y = self.n1(y + self.drop(self.sa(y, y, y, self_mask)))          # enmascarada (causal)
        y = self.n2(y + self.drop(self.ca(y, enc, enc, cross_mask)))     # atención cruzada
        return self.n3(y + self.drop(self.ff(y)))

class Transformer(nn.Module):
    def __init__(self, Vs, Vt, d=64, h=4, dff=128, n_layers=2, p=0.1):
        super().__init__()
        self.d = d
        self.src_emb, self.tgt_emb = nn.Embedding(Vs, d, padding_idx=PAD), nn.Embedding(Vt, d, padding_idx=PAD)
        self.pos = PositionalEncoding(d)
        self.enc_layers = nn.ModuleList(EncoderLayer(d, h, dff, p) for _ in range(n_layers))
        self.dec_layers = nn.ModuleList(DecoderLayer(d, h, dff, p) for _ in range(n_layers))
        self.out, self.drop = nn.Linear(d, Vt), nn.Dropout(p)

    def encode(self, src):
        mask = (src != PAD)[:, None, None, :]                                   # (B,1,1,S)
        x = self.drop(self.pos(self.src_emb(src) * math.sqrt(self.d)))
        for l in self.enc_layers: x = l(x, mask)
        return x, mask

    def decode(self, tgt, enc, src_mask):
        T = tgt.size(1)
        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=tgt.device))[None, None]   # no mirar al futuro
        self_mask = causal & (tgt != PAD)[:, None, None, :]
        y = self.drop(self.pos(self.tgt_emb(tgt) * math.sqrt(self.d)))
        for l in self.dec_layers: y = l(y, enc, self_mask, src_mask)
        return self.out(y)

    def forward(self, src, tgt):
        enc, src_mask = self.encode(src)
        return self.decode(tgt[:, :-1], enc, src_mask)                          # entrada del decoder: tgt sin último token

    @torch.no_grad()
    def translate(self, sentence, vocab_s, vocab_t, max_len=40):
        self.eval()
        src = torch.tensor([vocab_s.encode(sentence)]).to(DEVICE)
        enc, src_mask = self.encode(src)
        ys = torch.tensor([[SOS]]).to(DEVICE)
        for _ in range(max_len):
            nxt = self.decode(ys, enc, src_mask)[:, -1].argmax(-1, keepdim=True)
            if nxt.item() == EOS: break
            ys = torch.cat([ys, nxt], dim=1)
        self.decode(ys, enc, src_mask)                                          # pasada final para guardar atenciones
        return vocab_t.decode(ys[0, 1:].tolist()), ys

torch.manual_seed(SEED)
tfm = Transformer(len(src_vocab), len(tgt_vocab)).to(DEVICE)
print(f'Parámetros: {sum(p.numel() for p in tfm.parameters()):,}')

In [ ]:
tfm_losses = train_translator(tfm, epochs=300, lr=1e-3)

In [ ]:
show_translations(tfm, train_pairs[:4], 'Transformer — ENTRENAMIENTO')
show_translations(tfm, test_pairs, 'Transformer — NO VISTAS')

plt.figure(figsize=(8, 4))
plt.plot(s2s_losses, label='GRU + Bahdanau'); plt.plot(tfm_losses, label='Transformer')
plt.yscale('log'); plt.xlabel('Época'); plt.ylabel('Cross-entropy (log)')
plt.title('Convergencia en el corpus reducido'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Visualizando la atención cruzada por cabeza
Cada una de las 4 cabezas aprende una "forma" distinta de alinear las palabras. Mostramos la **última capa del decoder**.

In [ ]:
sent = "Remedios la bella subió al cielo en cuerpo y alma."
pred, ys = tfm.translate(sent, src_vocab, tgt_vocab)
cross = tfm.dec_layers[-1].ca.attn[0].cpu().numpy()          # (heads, T_generada, S)
src_tok = tokenize(sent) + ['<eos>']
# la fila t corresponde a la posición del decoder que GENERA el token t (su entrada es el token t-1)
tgt_tok = (pred.split() + ['<eos>'])[:cross.shape[1]]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, ax in enumerate(axes):
    plot_attention(cross[i], src_tok, tgt_tok, f'Cabeza {i+1}', ax)
plt.suptitle(f'Atención cruzada — "{pred}"', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Auto-atención del encoder (última capa): ¿qué palabras de la oración se relacionan entre sí?
_ = tfm.translate(sent, src_vocab, tgt_vocab)
sa = tfm.enc_layers[-1].sa.attn[0].mean(0).cpu().numpy()      # promedio de las cabezas
plt.figure(figsize=(6, 5))
plt.imshow(sa, cmap='magma')
plt.xticks(range(len(src_tok)), src_tok, rotation=60, ha='right'); plt.yticks(range(len(src_tok)), src_tok)
plt.title('Auto-atención del encoder (promedio de cabezas)'); plt.colorbar(); plt.tight_layout(); plt.show()

---
## Resumen y Preguntas para Discusión

| Modelo | Dato | Idea clave | Limitación observada |
|---|---|---|---|
| LSTM (S1) | Serie de temperatura | Celda de memoria con compuertas | El pronóstico recursivo acumula error |
| RNN/GRU/LSTM (S2) | Misma serie, `W=90` | Compuertas mitigan *vanishing gradients* | Procesamiento secuencial (lento) |
| GRU + Bahdanau (S3) | ES→EN | El decoder "consulta" palabras específicas del origen | Sigue siendo secuencial |
| Transformer (S4) | ES→EN | Atención en paralelo; sin recurrencia | Requiere muchos datos; memoriza con pocos |

**Preguntas:**
1. ¿Por qué se normaliza con estadísticas del conjunto de entrenamiento y no de toda la serie?
2. ¿Qué pasa con el error del pronóstico recursivo si aumentamos `H` a 90 días? ¿Cómo se podría mitigar (p. ej., salida multi-paso directa)?
3. Las oraciones no vistas se traducen mal. ¿Cuántos datos serían necesarios? ¿Qué papel juegan el *pre-entrenamiento* y la tokenización en subpalabras (BPE)?
4. ¿Por qué el decoder del Transformer necesita la máscara causal durante el entrenamiento, pero no se "nota" durante la inferencia?
5. Compara el número de parámetros y el tiempo de entrenamiento de S3 y S4. ¿Qué cambiaría con oraciones de 500 tokens?